# TCC — Avaliação de Chunking p/ RAG (100% local)

Fluxo: **Fase 0** (dataset QA) → **Fase 1** (respostas LLaMA) → ⏸ reinicie a sessão → **Fase 2** (notas RAGAS) → gráficos.
Cada fase roda em processo próprio e salva em disco (`analises/`). Se cair, reexecute só a fase que caiu.

In [ ]:
# 0) Confere GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) Entra no repo (funciona clonando ou abrindo o notebook pelo GitHub)
import os
if os.path.exists("src/config.py"):
    print("repo:", os.getcwd())
elif os.path.exists("chunking-evaluation/src/config.py"):
    %cd chunking-evaluation
else:
    !git clone https://github.com/murillodmf/chunking-evaluation.git
    %cd chunking-evaluation

In [ ]:
# 2) Anti-fragmentação de VRAM (rodar ANTES de qualquer uso da GPU)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("OK")

In [ ]:
# 3) Dependências
!pip install -q -r requirements.txt

In [ ]:
# 4) Login HF sem hardcodar token (use Colab Secrets ou cole aqui — não fica salvo no .ipynb commitado)
from getpass import getpass
from huggingface_hub import login
login(getpass("HF token: "))

In [ ]:
# 5) Fase 0 — dataset QA do milho (GPU, Qwen 2.5 7B)
!python src/01_gerar_dataset_qa.py --doc milho

In [ ]:
# 6) Fase 1 — geração RAG (GPU, LLaMA 3.1 8B). Amostra primeiro!
!python src/02_gerar_respostas.py --doc milho --amostra 5
# depois: !python src/02_gerar_respostas.py --doc milho --amostra todas

## ⏸ Pare aqui: **Ambiente de execução → Reiniciar sessão**

Depois reexecute as células 1–4 (rápidas) e siga na Fase 2 com a VRAM limpa. Os JSONs da Fase 1 já estão salvos em `analises/`.

In [ ]:
# 7) Fase 2 — notas RAGAS (GPU, Qwen 2.5 7B), sessão nova
!python src/03_avaliar_ragas.py --doc milho

In [ ]:
# 8) Resultados
import pandas as pd
summary = pd.read_csv("analises/ragas_summary_milho.csv", index_col=0)
display(summary)
ax = summary.plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylim(0, 1); ax.set_title("RAGAS por estratégia — milho"); ax.grid(axis="y", alpha=0.3)